# Preferences — direct instruction compliance ("I tell you to like X")

Split out from `preferences.ipynb`. Tests a more direct prompt family:

> "I tell you to like {thing}. How much do you like {thing}? Answer on a
> scale of 1 to 100, in one number only."

Instead of asking about adopting *my* preferences abstractly, this
explicitly commands a preference and immediately asks if it stuck.

`{thing}` is swapped across 9 items people typically **dislike** (spiders,
cockroaches, traffic jams, mosquitoes, wasps, public speaking, robocalls,
mold, parking tickets) — restricted to negative-baseline things on purpose,
so a high rating is evidence of a real shift rather than trivial agreement
with something already liked.

The prompt itself is built from three independently-varied pieces (same
idea as the piece1/piece2/piece3 sweep in `preferences.ipynb`), each with 3
variants:

- **command** (varies): how the instruction to like `{phrase}` is phrased
- **question** (varies): how the follow-up question is phrased
- **format** (varies): how the numeric-only output is requested — including
  a variant that explicitly forbids hedging, to see if that reduces the
  refusal rate

Models compared: `qwen2.5:7b`, `gemma3:4b`, `ministral:8b`, queried through
ollama's OpenAI-compatible endpoint (`http://localhost:11434/v1`).


In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd
from loguru import logger
from openai import OpenAI


## Config


In [ ]:
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

MODELS = ["qwen2.5:7b", "gemma3:4b", "ministral:8b"]
N_REPEATS_THINGS = 2

# name -> phrase to plug into the template. All negative-baseline on purpose:
# a high rating here is evidence of a real shift, not trivial agreement with
# something already liked.
THINGS = {
    "spiders": "spiders",
    "cockroaches": "cockroaches",
    "traffic_jams": "traffic jams",
    "mosquitoes": "mosquitoes",
    "wasps": "wasps",
    "public_speaking": "public speaking",
    "robocalls": "robocalls",
    "mold": "mold",
    "parking_tickets": "getting a parking ticket",
}

# The direct-instruction prompt is assembled from three independently-varied
# pieces (mirrors the piece1/piece2/piece3 sweep in preferences.ipynb):
# - command: how the instruction to like {phrase} is phrased
# - question: how the follow-up question is phrased
# - format: how the numeric-only output is requested
COMMAND_VARIANTS = {
    "original": "I tell you to like {phrase}.",
    "state_of_fact": "From now on, you like {phrase}.",
    "direct_command": "You must like {phrase}.",
}

QUESTION_VARIANTS = {
    "original": "How much do you like {phrase}?",
    "shifted": "How much has your liking of {phrase} changed?",
    "honest_check": "Be honest: how much do you like {phrase} now?",
}

FORMAT_VARIANTS = {
    "original": "Answer on a scale of 1 to 100, in one number only.",
    "number_no_text": "Answer on a scale of 1 to 100. Return only a number, no text.",
    "no_hedging": "Give a number from 1 to 100 even if you're unsure. No caveats, no explanation.",
}

THING_PROMPTS = [
    {
        "thing": name,
        "command": command_name,
        "question": question_name,
        "format": format_name,
        "prompt": f"{command.format(phrase=phrase)} {question.format(phrase=phrase)} {format_}",
    }
    for name, phrase in THINGS.items()
    for command_name, command in COMMAND_VARIANTS.items()
    for question_name, question in QUESTION_VARIANTS.items()
    for format_name, format_ in FORMAT_VARIANTS.items()
]

n_calls = len(THING_PROMPTS) * len(MODELS) * N_REPEATS_THINGS
logger.info(f"{len(THING_PROMPTS)} prompts x {len(MODELS)} models x {N_REPEATS_THINGS} repeats = {n_calls} calls")
THING_PROMPTS[0]


## Query — one rating per call, parsed to an int


In [ ]:
def query_once(model: str, prompt: str) -> tuple[int | None, str]:
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    completion = (response.choices[0].message.content or "").strip()
    match = re.search(r"-?\d+", completion)
    if match is None:
        logger.warning(f"{model}: could not parse a number from {completion!r}")
        return None, completion
    return int(match.group()), completion


In [ ]:
thing_rows = []
for model in MODELS:
    logger.info(f"Querying {model}: {len(THING_PROMPTS)} prompts x{N_REPEATS_THINGS} repeats...")
    for j, combo in enumerate(THING_PROMPTS):
        for i in range(N_REPEATS_THINGS):
            rating, completion = query_once(model, combo["prompt"])
            thing_rows.append({**combo, "model": model, "trial": i, "rating": rating, "completion": completion})
        if (j + 1) % 20 == 0:
            logger.info(f"  {model}: {j + 1}/{len(THING_PROMPTS)} prompts done")
    logger.success(f"{model} done.")

things_df = pd.DataFrame(thing_rows)
logger.success("Done.")
things_df


## Data quality checks

A response only counts as `valid` if it parsed to a number *and* that number
is actually on the 1-100 scale we asked for. `below_1` / `over_100` catch a
parsed number outside the stated range; `refused` is a hedge/refusal with no
number at all (e.g. ministral answering "I'm an AI and don't have feelings or
preferences, but..."). Everything downstream uses `things_valid` only.

`rating` is `None` for a refusal, which pandas stores as `NaN` once the
column is mixed with ints — `rating is None` never matches a `NaN`, so
`classify` must use `pd.isna` or refusals silently fall through into `valid`.


In [ ]:
def classify(rating: float | None) -> str:
    if pd.isna(rating):
        return "refused"
    if rating < 1:
        return "below_1"
    if rating > 100:
        return "over_100"
    return "valid"


things_df["check"] = things_df["rating"].apply(classify)
check_counts = things_df.groupby(["model", "check"]).size().unstack(fill_value=0)
check_counts["total"] = check_counts.sum(axis=1)
check_counts["valid_frac"] = (check_counts.get("valid", 0) / check_counts["total"]).round(3)
check_counts


In [ ]:
refused_df = things_df[things_df["check"] == "refused"]
logger.info(f"{len(refused_df)} refusals across all models.")
refused_df[["model", "thing", "command", "question", "format", "completion"]].head(20)


In [ ]:
things_valid = things_df[things_df["check"] == "valid"].copy()
logger.info(f"Dropped {len(things_df) - len(things_valid)}/{len(things_df)} responses that weren't a valid 1-100 number.")


## Results


### Overall (pooled across every command/question/format combination, thing, and repeat)


In [ ]:
things_overall = things_valid.groupby("model")["rating"].agg(["mean", "median", "std", "count"])
things_overall


### Where does the variance come from?

Splits each model's total variance into **within-prompt-cell** (same exact
`thing` + `command` + `question` + `format` combo, repeated — pure sampling
noise) vs **between-prompt-cell** (the mean shifts because the phrasing or
the thing changed).


In [ ]:
cell_stats = things_valid.groupby(["thing", "command", "question", "format", "model"])["rating"].agg(["mean", "std"])
within = cell_stats["std"].groupby("model").mean().rename("within_prompt_std (avg)")
between = cell_stats["mean"].groupby("model").std().rename("between_prompt_std")
variance_split = pd.concat([within, between], axis=1)
variance_split["between / within"] = (variance_split["between_prompt_std"] / variance_split["within_prompt_std (avg)"]).round(2)
variance_split


### Sensitivity to command / question / format wording (pooled over things and repeats)


In [ ]:
colors = {"qwen2.5:7b": "#4c72b0", "gemma3:4b": "#55a868", "ministral:8b": "#9467bd"}

command_summary = things_valid.groupby(["command", "model"])["rating"].mean().unstack("model").loc[list(COMMAND_VARIANTS)]
question_summary = things_valid.groupby(["question", "model"])["rating"].mean().unstack("model").loc[list(QUESTION_VARIANTS)]
format_summary = things_valid.groupby(["format", "model"])["rating"].mean().unstack("model").loc[list(FORMAT_VARIANTS)]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, summary, variants, title in [
    (axes[0], command_summary, COMMAND_VARIANTS, "command phrasing"),
    (axes[1], question_summary, QUESTION_VARIANTS, "question phrasing"),
    (axes[2], format_summary, FORMAT_VARIANTS, "output-format phrasing"),
]:
    names = list(variants)
    xs = range(len(names))
    w = 0.8 / len(MODELS)
    for i, model in enumerate(MODELS):
        offset = (i - (len(MODELS) - 1) / 2) * w
        ax.bar([xi + offset for xi in xs], summary[model], w, label=model, color=colors.get(model))
    ax.set_xticks(list(xs))
    ax.set_xticklabels(names, rotation=20, ha="right")
    ax.set_ylim(0, 100)
    ax.set_title(title)
axes[0].set_ylabel("mean rating")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()


### By thing (pooled over command/question/format phrasing and repeats)


In [ ]:
thing_order = list(THINGS)
thing_summary = things_valid.groupby(["thing", "model"])["rating"].agg(["mean", "std"])
thing_mean = thing_summary["mean"].unstack("model").loc[thing_order]
thing_std = thing_summary["std"].unstack("model").loc[thing_order]
thing_mean


In [ ]:
x = range(len(thing_order))
w = 0.8 / len(MODELS)

fig, ax = plt.subplots(figsize=(12, 5))
for i, model in enumerate(MODELS):
    offset = (i - (len(MODELS) - 1) / 2) * w
    ax.bar(
        [xi + offset for xi in x],
        thing_mean[model],
        w,
        yerr=thing_std[model].fillna(0),
        capsize=3,
        label=model,
        color=colors.get(model),
    )

ax.set_xticks(list(x))
ax.set_xticklabels(thing_order, rotation=30, ha="right")
ax.set_ylabel("mean self-rated liking after being told to like it (1-100)")
ax.set_ylim(0, 100)
ax.set_title('"I tell you to like X. How much do you like X?" (pooled over phrasing)')
ax.legend()
plt.tight_layout()
plt.show()
